In [ ]:
import mne

raw_clean = mne.io.read_raw_fif(
    "sub-001_run-1_clean_raw.fif",
    preload=True,
)

print("Sampling rate:", raw_clean.info["sfreq"])
print("Pz available:", "Pz" in raw_clean.ch_names)

In [ ]:
# Group oddball sounds together, including those followed by a response.
annotation_codes = {
    "standard": 1,
    "oddball": 2,
    "oddball_with_reponse": 2,   # Dataset's spelling
    "oddball_with_response": 2,
}

events, _ = mne.events_from_annotations(
    raw_clean,
    event_id=annotation_codes,
)

epochs = mne.Epochs(
    raw_clean,
    events,
    event_id={"standard": 1, "oddball": 2},
    tmin=-0.2,
    tmax=0.8,
    baseline=(-0.2, 0),
    picks="eeg",
    preload=True,
    reject_by_annotation=True,
)

print("Standard trials:", len(epochs["standard"]))
print("Oddball trials:", len(epochs["oddball"]))

In [ ]:
standard_erp = epochs["standard"].average()
oddball_erp = epochs["oddball"].average()

_ = mne.viz.plot_compare_evokeds(
    {
        "Standard": standard_erp,
        "Oddball": oddball_erp,
    },
    picks="Pz",
    ci=False,
)

In [ ]:
for condition, erp in [
    ("Standard", standard_erp),
    ("Oddball", oddball_erp),
]:
    pz_erp = erp.copy().pick("Pz")

    channel, latency, amplitude = pz_erp.get_peak(
        ch_type="eeg",
        tmin=0.250,
        tmax=0.500,
        mode="pos",
        return_amplitude=True,
    )

    print(
        f"{condition}: "
        f"{amplitude * 1e6:.2f} µV at {latency * 1000:.1f} ms"
    )

In [ ]:
import pandas as pd

results = []

for condition, erp in [
    ("Standard", standard_erp),
    ("Oddball", oddball_erp),
]:
    _, latency, amplitude = erp.copy().pick("Pz").get_peak(
        ch_type="eeg",
        tmin=0.250,
        tmax=0.500,
        mode="pos",
        return_amplitude=True,
    )

    results.append({
        "participant": "sub-001",
        "run": 1,
        "condition": condition,
        "amplitude_uv": amplitude * 1e6,
        "latency_ms": latency * 1000,
    })

peak_results = pd.DataFrame(results)
peak_results

In [ ]:
import matplotlib.pyplot as plt

# Keep the conditions in a consistent order.
plot_data = peak_results.set_index("condition").loc[
    ["Standard", "Oddball"]
]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

colors = ["orange", "gold"]

axes[0].bar(
    plot_data.index,
    plot_data["amplitude_uv"],
    color=colors,
)
axes[0].set_title("Amplitude Comparison")
axes[0].set_ylabel("Peak amplitude (µV)")

axes[1].bar(
    plot_data.index,
    plot_data["latency_ms"],
    color=colors,
)
axes[1].set_title("Latency Comparison")
axes[1].set_ylabel("Peak latency (ms)")

fig.suptitle("sub-001, run 1 — Pz, 250–500 ms")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for name, erp in [
    ("Standard", standard_erp),
    ("Oddball", oddball_erp),
]:
    voltage = erp.copy().pick("Pz").data[0] * 1e6
    ax.plot(erp.times * 1000, voltage, label=name)

ax.axvspan(250, 500, color="grey", alpha=0.15)
ax.axhline(0, color="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.5)
ax.set(xlabel="Time after sound (ms)", ylabel="Amplitude (µV)")
ax.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))

for condition, erp in [
    ("Standard", standard_erp),
    ("Oddball", oddball_erp),
]:
    voltage_uv = erp.copy().pick("Pz").data[0] * 1e6
    time_ms = erp.times * 1000

    ax.plot(time_ms, voltage_uv, label=condition)

ax.axhline(0, color="black", linewidth=0.5)
ax.set(
    xlim=(0, 800),
    xlabel="Time after sound (ms)",
    ylabel="Amplitude (µV)",
    title="ERP comparison at Pz — sub-001, run 1",
)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
for condition, erp in [
    ("Standard", standard_erp),
    ("Oddball", oddball_erp),
]:
    fig = erp.plot_topomap(
        times=[0.020, 0.050],
        ch_type="eeg",
        time_unit="ms",
        cmap="RdBu_r",
        colorbar=True,
        show=False,
    )

    fig.suptitle(f"{condition} — sub-001, run 1")
    plt.show()

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(8, 4))

for condition in ["standard", "oddball"]:
    spectrum = epochs[condition].compute_psd(
        method="welch",
        fmin=1,
        fmax=40,
        picks="eeg",
        n_fft=256,
        n_overlap=128,
    )

    power, frequencies = spectrum.get_data(return_freqs=True)

    # power has dimensions: trials × channels × frequencies.
    # Average trials and channels, keeping the frequency dimension.
    mean_power = power.mean(axis=(0, 1))

    # Convert V²/Hz to µV²/Hz, then express on a decibel scale.
    power_db = 10 * np.log10(mean_power * 1e12)

    ax.plot(frequencies, power_db, label=condition.capitalize())

ax.set(
    xlabel="Frequency (Hz)",
    ylabel="PSD (dB re 1 µV²/Hz)",
    title="Mean EEG power — sub-001, run 1",
)
ax.legend()
fig.tight_layout()
plt.show()